In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [58]:
def load_data(file_path):
    if file_path.lower().endswith(".csv"):
        return pd.read_csv(file_path)

    elif file_path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(file_path)

    else:
        raise ValueError(
            "Unsupported file format. Use CSV or Excel."
        )

In [59]:
df = load_data("../data/raw/Online Retail.xlsx")

In [60]:
df_original = df.copy()

In [61]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn Names:")
print(list(df.columns))

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Rows: 541909
Columns: 8

Column Names:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Data Types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

Missing Values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Duplicate Rows:
5268


In [62]:
def clean_column_names(df):

    df = df.copy()

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^\w]", "", regex=True)
    )

    return df

In [63]:
df = clean_column_names(df)

In [64]:
df = df.dropna(how="all")

In [65]:
df = df.dropna(axis=1, how="all")

In [66]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 5268


In [67]:
df = df.drop_duplicates()

In [68]:
print("Duplicates after cleaning:", df.duplicated().sum())

Duplicates after cleaning: 0


In [69]:
text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

print(list(text_columns))

['invoiceno', 'stockcode', 'description', 'country']


In [70]:
for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

In [71]:
missing_report = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage":
        (df.isnull().sum() / len(df)) * 100
})

missing_report = missing_report.sort_values(
    "missing_percentage",
    ascending=False
)

missing_report

,missing_count,missing_percentage
customerid,135037,25.163377
description,1454,0.270945
stockcode,0,0.000000
invoiceno,0,0.000000
quantity,0,0.000000
invoicedate,0,0.000000
unitprice,0,0.000000
country,0,0.000000


In [72]:
numeric_columns = df.select_dtypes(
    include=np.number
).columns

In [73]:
categorical_columns = df.select_dtypes(
    include=["object", "string"]
).columns

In [75]:
for col in numeric_columns:

    if df[col].isnull().sum() > 0:

        df[col] = df[col].fillna(
            df[col].median()
        )

In [77]:
for col in categorical_columns:

    if df[col].isnull().sum() > 0:

        mode = df[col].mode()

        if len(mode) > 0:
            df[col] = df[col].fillna(mode.iloc[0])

In [78]:
numeric_columns = df.select_dtypes(
    include=np.number
).columns

print(list(numeric_columns))

['quantity', 'unitprice', 'customerid']


In [79]:
df[numeric_columns].describe()

,quantity,unitprice,customerid
count,536641.000000,536641.000000,536641.000000
mean,9.620029,4.632656,15246.898157
std,219.130156,97.233118,1483.931554
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,14367.000000
50%,3.000000,2.080000,15145.000000
75%,10.000000,4.130000,16241.000000
max,80995.000000,38970.000000,18287.000000


In [80]:
def detect_date_columns(df):

    date_columns = []

    for col in df.columns:

        if df[col].dtype == "object":

            converted = pd.to_datetime(
                df[col],
                errors="coerce"
            )

            if converted.notna().mean() >= 0.8:
                date_columns.append(col)

    return date_columns

In [81]:
date_columns = detect_date_columns(df)

print("Possible date columns:")
print(date_columns)

Possible date columns:
[]


In [82]:
for col in date_columns:

    df[col] = pd.to_datetime(
        df[col],
        errors="coerce"
    )

In [83]:
def detect_outliers(df):

    outlier_report = {}

    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns

    for col in numeric_columns:

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        count = (
            (df[col] < lower) |
            (df[col] > upper)
        ).sum()

        outlier_report[col] = count

    return outlier_report

In [84]:
outliers = detect_outliers(df)

outliers

{'quantity': np.int64(58501),
 'unitprice': np.int64(39450),
 'customerid': np.int64(0)}

In [85]:
original_rows = len(df_original)

In [86]:
cleaned_rows = len(df)

print("Original rows:", original_rows)
print("Cleaned rows:", cleaned_rows)
print("Rows removed:", original_rows - cleaned_rows)

Original rows: 541909
Cleaned rows: 536641
Rows removed: 5268


In [87]:
print("Final Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nData Types:")
print(df.dtypes)

Final Shape: (536641, 8)

Missing Values:
invoiceno      0
stockcode      0
description    0
quantity       0
invoicedate    0
unitprice      0
customerid     0
country        0
dtype: int64

Duplicate Rows:
0

Data Types:
invoiceno              string
stockcode              string
description            string
quantity                int64
invoicedate    datetime64[us]
unitprice             float64
customerid            float64
country                string
dtype: object


In [88]:
df.to_csv(
    "../data/processed/cleaned_data.csv",
    index=False
)